# Module 2b — PDF Parsing

> **Chapter 2, Module B of 14** · [chapter index](../README.md)
> **Prerequisites:** [2a — Why Ingestion Matters](../2a.%20Why%20ingestion%20and%20canonical%20documents/)
> **Next:** [2c — OCR & Scanned Documents](../2c.%20OCR%20and%20scanned%20documents/)


## The one idea

**PDF is a presentation format, not a semantic document format.** It stores *where ink goes*,
not *what the document means*. Every piece of structure your parser reports — paragraph,
table, reading order — is **inferred from coordinates**, and inference fails.

We will prove this on three real PDFs, each of which breaks a different assumption:

| Document | Breaks |
|---|---|
| **BERT paper** (arXiv, true 2-column) | Reading order — columns interleave |
| **RAG paper** (arXiv, single-column) | The *column-detector itself* — a control case |
| **IRS Form W-4** (AcroForm) | The assumption that text extraction sees everything |
| **GDPR** (88 pages of law) | The assumption that extracted text is clean |

**By the end you will be able to:**

1. Explain why PDF parsing is hard (a top-5 interview question) with evidence
2. Classify a PDF as digital / scanned / complex-layout before choosing a strategy
3. Extract text while preserving page provenance
4. Detect and fix the two-column reading-order problem using coordinates
5. Know when your layout heuristic is the thing that is wrong
6. Recognise content that text extraction silently misses

## Setup

In [1]:
# `sys` lets us add the project root to Python's import path.
import sys

# The notebook sits two levels below the project root (chapter/module/notebook).
sys.path.append("../..")

# `Path` builds cross-platform filesystem paths.
from pathlib import Path

# The corpus helper downloads real public documents and caches them locally.
from utils.corpus import fetch, describe, CORPUS

# Confirm the helper loaded and show how many documents are catalogued.
print(f"Corpus helper loaded - {len(CORPUS)} real documents available.")

Corpus helper loaded - 9 real documents available.


In [2]:
# Define a small helper that reports whether an optional library is installed.
def try_import(module_name, friendly_name, install_hint):

    # Attempt the import inside a try block so a missing package is not fatal.
    try:
        module = __import__(module_name)

    # If the package is absent, report how to install it and return None.
    except ImportError:
        print(f"  [ ] {friendly_name:<16} missing  -  pip install {install_hint}")
        return None

    # On success, report availability and hand back the module object.
    print(f"  [x] {friendly_name:<16} available")
    return module


# Announce what we are checking.
print("Required library:")

# PyMuPDF was historically imported as `fitz`; recent versions prefer `pymupdf`
# and warn on the old name. Try the modern name first, then fall back.
pymupdf = (try_import("pymupdf", "PyMuPDF", "pymupdf")
           or try_import("fitz", "PyMuPDF (legacy)", "pymupdf"))

Required library:
  [x] PyMuPDF          available


In [3]:
# Only fetch if PyMuPDF is available to read the files.
if pymupdf is not None:

    # Download (or load from cache) the three real PDFs this module uses.
    print("Real PDFs for this module:")
    rag_paper_path = fetch("rag_paper")     # single-column  (control case)
    bert_path = fetch("bert_paper")         # genuinely two-column
    w4_path = fetch("irs_w4")               # AcroForm with hidden fields
    gdpr_path = fetch("gdpr_pdf")           # 88 pages of real legislation

Real PDFs for this module:
  cached  rag_paper_lewis_2020.pdf  (865 KB)
  cached  bert_devlin_2019.pdf  (757 KB)
  cached  irs_form_w4.pdf  (204 KB)
  cached  gdpr_regulation_2016_679.pdf  (959 KB)


# 1. Why PDF is hard

A human sees:

```
TITLE

Introduction

This regulation applies to...

Table 1
...
```

Internally the PDF may store:

```
character x=120 y=515
character x=131 y=515
character x=144 y=515
...
```

There may be **no true concept of**: paragraph · section · table · reading order.

Let's look at what is actually stored, using the real RAG paper.

In [4]:
# Only run if PyMuPDF is available.
if pymupdf is not None:

    # Open the academic paper.
    document = pymupdf.open(rag_paper_path)

    # Load the title page.
    page = document.load_page(0)

    # get_text("words") returns one tuple per word, WITH its bounding box.
    # Each tuple is (x0, y0, x1, y1, word, block_no, line_no, word_no).
    words = page.get_text("words")

    # Show the raw positional reality underneath the "text" we normally see.
    print(f"Page 1 contains {len(words)} positioned words. First five:\n")

    # Print the coordinates of the first few words.
    for x0, y0, x1, y1, word, *_ in words[:5]:
        print(f"  x={x0:6.1f}  y={y0:6.1f}   {word!r}")

    # Close the document to release the file handle.
    document.close()

Page 1 contains 401 positioned words. First five:

  x= 170.8  y=  99.8   'Retrieval-Augmented'
  x= 332.5  y=  99.8   'Generation'
  x= 420.0  y=  99.8   'for'
  x= 186.5  y= 119.8   'Knowledge-Intensive'
  x= 345.3  y= 119.8   'NLP'


**That is the ground truth of a PDF**: floating-point coordinates and glyphs. Everything
else — "this is a paragraph", "this is a heading", "read this column first" — is
reconstruction.

## 1.1 Three broad PDF types — identify before you extract

```
PDF
│
├── Digital / text-based      →  extract text directly
│
├── Scanned / image-based     →  needs OCR (module 2c)
│
└── Mixed / complex-layout    →  needs layout-aware parsing
```

Choosing wrong is expensive in both directions: running OCR on a digital PDF wastes money
(see [module 2l](../2l.%20Production%20architecture/)), and running a text parser on a scan
silently returns nothing.

Let's build the classifier.

In [5]:
# Define a function that classifies a PDF before we choose an extraction strategy.
def classify_pdf(pdf_path, min_chars_per_page=50):

    # Open the document.
    document = pymupdf.open(pdf_path)

    # Count how many pages carry a usable text layer.
    pages_with_text = 0

    # Track total extracted characters, for reporting.
    total_characters = 0

    # Examine every page.
    for page_number in range(len(document)):

        # Extract the text layer of this page.
        text = document.load_page(page_number).get_text("text").strip()

        # Accumulate the character count.
        total_characters += len(text)

        # A page with almost no text is very likely an image.
        if len(text) >= min_chars_per_page:
            pages_with_text += 1

    # Remember the page count before closing the file.
    page_count = len(document)

    # Release the file handle.
    document.close()

    # Guard against a zero-page file.
    if page_count == 0:
        return {"type": "EMPTY", "pages": 0, "text_ratio": 0.0, "avg_chars": 0}

    # What fraction of pages have a real text layer.
    text_ratio = pages_with_text / page_count

    # Mostly text -> digital; mostly image -> scanned; otherwise mixed.
    if text_ratio >= 0.9:
        pdf_type = "DIGITAL"
    elif text_ratio <= 0.1:
        pdf_type = "SCANNED"
    else:
        pdf_type = "MIXED"

    # Return the diagnosis plus the evidence behind it.
    return {
        "type": pdf_type,
        "pages": page_count,
        "text_ratio": text_ratio,
        "avg_chars": total_characters // page_count,
    }


# Only run if PyMuPDF is available.
if pymupdf is not None:

    # Classify all three real PDFs.
    for label, path in [("RAG paper", rag_paper_path),
                        ("BERT paper", bert_path),
                        ("IRS W-4", w4_path),
                        ("GDPR", gdpr_path)]:

        # Run the classifier.
        result = classify_pdf(path)

        # Report the verdict with its supporting numbers.
        print(f"  {label:12s} {result['type']:8s} "
              f"{result['pages']:3d} pages, "
              f"{result['text_ratio']:.0%} with text, "
              f"~{result['avg_chars']} chars/page")

  RAG paper    DIGITAL   19 pages, 100% with text, ~3634 chars/page
  BERT paper   DIGITAL   16 pages, 100% with text, ~4006 chars/page
  IRS W-4      DIGITAL    5 pages, 100% with text, ~5217 chars/page
  GDPR         DIGITAL   88 pages, 100% with text, ~4095 chars/page


All three are `DIGITAL` — so text extraction will work. But *working* and *being correct*
are very different things, as the next section shows.

# 2. Extraction that preserves provenance

The single most common beginner mistake:

```python
full_text = "".join(all_pages)   # DON'T
```

You lose: **page boundaries · citations · headers · sections · references.**

Page number is valuable metadata for **citations, debugging and source linking**. Once you
throw it away at stage one, no later stage can recover it.

In [6]:
# Define a function for extracting text from a digital PDF, page by page.
def extract_pdf_pages(pdf_path, max_pages=None):

    # Open the PDF document from disk.
    document = pymupdf.open(pdf_path)

    # Collect one record per page.
    page_texts = []

    # Decide how many pages to read (useful for sampling an 88-page document).
    page_limit = len(document) if max_pages is None else min(max_pages, len(document))

    # Iterate through the requested page range.
    for page_number in range(page_limit):

        # Load the page object at this index.
        page = document.load_page(page_number)

        # Extract the plain text of the page in the parser's inferred reading order.
        text = page.get_text("text")

        # Store the page number alongside its text, so provenance survives.
        page_texts.append({
            "page_number": page_number + 1,   # human-facing pages start at 1
            "text": text,
        })

    # Close the file handle to release the OS resource.
    document.close()

    # Return the list of per-page records.
    return page_texts


# Only run if PyMuPDF is available.
if pymupdf is not None:

    # Extract the first three pages of the real GDPR regulation.
    gdpr_pages = extract_pdf_pages(gdpr_path, max_pages=3)

    # Show the opening of page 1 - real EU legislation.
    print("GDPR page 1, first 400 characters:\n")
    print(gdpr_pages[0]["text"][:400])

GDPR page 1, first 400 characters:

I 
(Legislative acts) 
REGULATIONS 
REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL 
of 27 April 2016 
on the protection of natural persons with regard to the processing of personal data and on the free 
movement of such data, and repealing Directive 95/46/EC (General Data Protection Regulation) 
(Text with EEA relevance) 
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROP


Look at the top of that output: `L 119/1`, `EN`, `Official Journal of the European Union`,
`4.5.2016`. That is **real boilerplate**, on a real document, before we have written a single
cleaning rule. [Module 2g](../2g.%20Cleaning%20and%20normalization/) removes it — and discovers a
trap while doing so.

### A citation-ready extraction

Because we kept `page_number`, we can produce a grounded citation for any passage.

In [7]:
# Only run if PyMuPDF is available.
if pymupdf is not None:

    # Search the real GDPR text for the article that defines the right to erasure.
    document = pymupdf.open(gdpr_path)

    # Track where we find the phrase.
    found_page = None

    # Scan every page for the phrase.
    for page_number in range(len(document)):

        # Extract this page's text.
        text = document.load_page(page_number).get_text("text")

        # Look for the well-known "right to be forgotten" heading.
        if "right to be forgotten" in text.lower():

            # Record the human-facing page number and stop.
            found_page = page_number + 1
            break

    # Close the document.
    document.close()

    # Report a real, verifiable citation.
    if found_page:
        print(f"Found 'right to be forgotten' on page {found_page}")
        print(f"Citation: GDPR (Regulation (EU) 2016/679), page {found_page}")
    else:
        print("Phrase not found - the document edition may differ.")

Found 'right to be forgotten' on page 13
Citation: GDPR (Regulation (EU) 2016/679), page 13


That is a **real citation to a real page of real legislation**, produced entirely by
preserving one integer through the extraction stage. Throw away `page_number` at stage one
and this is impossible forever after.

# 3. The reading-order problem

This is where PDFs bite hardest.

Consider two columns:

```
COLUMN A              COLUMN B

A1                    B1
A2                    B2
A3                    B3
```

| | Sequence |
|---|---|
| **Correct reading order** | A1 A2 A3 B1 B2 B3 |
| **Naive parser output** | A1 B1 A2 B2 A3 B3 |

Semantics destroyed — sentences from one column interleave with the other.

## 3.1 Detecting columns from coordinates

Layout-aware parsers use bounding-box coordinates $(x_1, y_1, x_2, y_2)$ for each text
block. If block $x$-positions cluster into two groups, you have two columns.

But **first you must establish whether the document has columns at all.** Let's build a
detector that answers that honestly rather than assuming.

In [8]:
# Define a function that decides whether a page is genuinely two-column.
def detect_layout(page, fullwidth_ratio=0.6):

    # Get every text block with its bounding box.
    blocks = [b for b in page.get_text("blocks") if b[6] == 0 and b[4].strip()]

    # A page with almost no text tells us nothing.
    if len(blocks) < 4:
        return {"layout": "UNKNOWN", "reason": "too few blocks"}

    # The vertical line that would separate two columns.
    midpoint = page.rect.width / 2

    # Count blocks starting on each side of that line.
    left_count = sum(1 for b in blocks if b[0] < midpoint)
    right_count = len(blocks) - left_count

    # Count blocks that span most of the page width - these CROSS any column divide.
    fullwidth_count = sum(
        1 for b in blocks
        if (b[2] - b[0]) > page.rect.width * fullwidth_ratio
    )

    # Many full-width blocks means the page is not cleanly two-column.
    fullwidth_fraction = fullwidth_count / len(blocks)

    # A real two-column page has blocks on BOTH sides and few full-width spans.
    balanced = min(left_count, right_count) / max(left_count, right_count, 1)

    # Decide, and report the evidence so the verdict is auditable.
    if balanced > 0.5 and fullwidth_fraction < 0.2:
        layout = "TWO_COLUMN"
    elif right_count == 0 or fullwidth_fraction > 0.3:
        layout = "SINGLE_COLUMN"
    else:
        layout = "MIXED"

    # Return the diagnosis together with the numbers behind it.
    return {
        "layout": layout,
        "blocks": len(blocks),
        "left": left_count,
        "right": right_count,
        "balance": balanced,
        "fullwidth_frac": fullwidth_fraction,
    }


# Only run if PyMuPDF is available.
if pymupdf is not None:

    # Compare a body page from each paper.
    for label, path in [("RAG paper", rag_paper_path), ("BERT paper", bert_path)]:

        # Open the document.
        document = pymupdf.open(path)

        # Page 3 (index 2) is body text in both papers.
        result = detect_layout(document.load_page(2))

        # Report the verdict with its evidence.
        print(f"  {label:12s} {result['layout']:14s} "
              f"L={result['left']:3d} R={result['right']:3d}  "
              f"balance={result['balance']:.2f}  "
              f"full-width={result['fullwidth_frac']:.0%}")

        # Close the document.
        document.close()

  RAG paper    MIXED          L= 28 R=  7  balance=0.25  full-width=26%
  BERT paper   TWO_COLUMN     L= 15 R= 16  balance=0.94  full-width=3%


> **This is the result that matters.** The RAG paper — despite being an arXiv NLP paper,
> which *looks* like it should be two-column — is **single-column**. The BERT paper is
> genuinely two-column.
>
> Had we assumed "academic paper ⇒ two columns" and applied a midpoint split to the RAG
> paper, we would have shredded its equations and figure captions into a meaningless order
> while believing we had *fixed* the reading order.

**The heuristic must be verified, not assumed.** That is the real lesson, and it only shows
up when you run the code on real documents.

## 3.2 Now fix the document that actually needs fixing

In [ ]:
# Define a function that reconstructs two-column reading order from coordinates.
def extract_two_column_order(page):

    # Get all blocks with their bounding boxes.
    blocks = [b for b in page.get_text("blocks") if b[6] == 0 and b[4].strip()]

    # The vertical line that separates the two columns.
    midpoint = page.rect.width / 2

    # Blocks whose left edge is in the left half belong to column one.
    left = [b for b in blocks if b[0] < midpoint]

    # Everything else belongs to column two.
    right = [b for b in blocks if b[0] >= midpoint]

    # Within each column, read top to bottom - that is what y0 orders by.
    left.sort(key=lambda b: b[1])
    right.sort(key=lambda b: b[1])

    # Correct reading order is the WHOLE left column, then the whole right column.
    return [b[4].strip().replace("\n", " ") for b in left + right]


# Only run if PyMuPDF is available.
if pymupdf is not None:

    # Open the genuinely two-column BERT paper.
    document = pymupdf.open(bert_path)

    # Use a body page.
    page = document.load_page(2)

    # Strategy A: whatever the library's default block order gives us.
    default_blocks = [b for b in page.get_text("blocks") if b[6] == 0 and b[4].strip()]

    # Strategy B: our explicit column-aware reconstruction.
    midpoint = page.rect.width / 2
    left = sorted([b for b in default_blocks if b[0] < midpoint], key=lambda b: b[1])
    right = sorted([b for b in default_blocks if b[0] >= midpoint], key=lambda b: b[1])
    column_blocks = left + right

    # The x-coordinate sequence reveals the problem instantly.
    print("Default block order, x-positions:")
    print("  ", [round(b[0]) for b in default_blocks[:12]])

    print("\nColumn-aware order, x-positions:")
    print("  ", [round(b[0]) for b in column_blocks[:12]])

    # Close the document.
    document.close()

**Read those two number sequences.**

The default order jumps between roughly 370 and 90 — it is bouncing **between the right and
left columns**, block by block. The column-aware order stays in the low numbers first
(finishing the left column completely) before moving right.

That bouncing is what interleaves two unrelated sentences into a single chunk.

## 3.3 A diagnostic you can run on any corpus

You will not eyeball coordinates for ten million documents, so you need an automatic signal.

A reasonable first attempt: count blocks that **start mid-sentence** (a lowercase first
letter). Interleaved columns ought to produce many of them.

Run it, then check whether it actually discriminates.

In [ ]:
# Define a heuristic that flags likely reading-order corruption.
def reading_order_suspicious(block_texts, threshold=0.5, min_blocks=5):

    # Too few blocks to judge reliably.
    if len(block_texts) < min_blocks:
        return False, "too few blocks to assess"

    # Count blocks that begin with a lowercase letter. One or two is normal
    # (a column continuing a sentence), but a HIGH rate means interleaving.
    lowercase_starts = sum(
        1 for text in block_texts
        if text.strip() and text.strip()[0].islower()
    )

    # Express it as a fraction of all blocks.
    ratio = lowercase_starts / len(block_texts)

    # Above the threshold, sentences are probably being cut across columns.
    if ratio > threshold:
        return True, f"{ratio:.0%} of blocks start mid-sentence"

    # Otherwise the ordering looks plausible.
    return False, f"{ratio:.0%} of blocks start mid-sentence"


# Only run if PyMuPDF is available.
if pymupdf is not None:

    # Reopen BERT to compare both orderings under the same diagnostic.
    document = pymupdf.open(bert_path)
    page = document.load_page(2)

    # Rebuild both orderings.
    blocks = [b for b in page.get_text("blocks") if b[6] == 0 and b[4].strip()]
    midpoint = page.rect.width / 2
    left = sorted([b for b in blocks if b[0] < midpoint], key=lambda b: b[1])
    right = sorted([b for b in blocks if b[0] >= midpoint], key=lambda b: b[1])

    # Extract just the text of each, in each order.
    default_texts = [b[4].strip().replace("\n", " ") for b in blocks]
    column_texts = [b[4].strip().replace("\n", " ") for b in left + right]

    # Run the diagnostic on both.
    for label, texts in [("default order", default_texts), ("column order", column_texts)]:
        suspicious, reason = reading_order_suspicious(texts)
        flag = "SUSPICIOUS" if suspicious else "ok        "
        print(f"  {label:14s} {flag}  ({reason})")

    # Close the document.
    document.close()

### The diagnostic is weaker than it looks

Both orderings score the same, so on this document the heuristic **fails to discriminate** —
even though we proved with coordinates that the orderings genuinely differ.

Why? PyMuPDF returns whole *paragraph* blocks, and a paragraph usually starts with a capital
letter regardless of which column it came from. The lowercase signal only fires when blocks
get split mid-sentence, which happens with weaker extractors or line-level extraction.

**Do not discard the idea — understand its domain.** Stronger signals for production:

| Signal | How it works |
|---|---|
| **Coordinate audit** | Compare `x0` sequences, as we did above. Cheap and decisive. |
| **Layout classification** | Run `detect_layout` first; apply column logic only to `TWO_COLUMN` pages. |
| **Sentence continuity** | Check whether a block ends in terminal punctuation before the next begins. |
| **LM perplexity** | Correctly ordered text scores far lower perplexity. Accurate, but expensive. |

This is what an honest heuristic looks like: it has a domain where it works, and you should
know where that domain ends before trusting it on ten million files.


## 3.4 What full layout-aware parsing reconstructs

A complete layout parser identifies regions such as: Title · Heading · Paragraph · List ·
Table · Image · Caption · Header · Footer · Equation.

```
            PDF PAGE

┌──────────────────────────┐
│ [TITLE]                  │
├──────────────────────────┤
│ [PARAGRAPH]              │
│                          │
├──────────────────────────┤
│ [TABLE]                  │
│                          │
├──────────────────────────┤
│ [FIGURE]     [CAPTION]   │
└──────────────────────────┘
```

This makes later chunking dramatically smarter — you can chunk *by section* rather than by
character count.

You do not always need a full layout model. PyMuPDF exposes font size **and font weight**,
which is enough on most documents. Let's try size first, on real legislation.

In [ ]:
# Define a function that infers headings from font size alone.
def detect_headings_by_size(page, size_threshold_ratio=1.15):

    # get_text("dict") gives the full structure: blocks -> lines -> spans.
    structure = page.get_text("dict")

    # Collect (size, text) for every span so we can find the body-text size.
    spans = []

    # Walk the nested structure.
    for block in structure["blocks"]:

        # Skip image blocks, which have no "lines" key.
        if "lines" not in block:
            continue

        # Each line contains one or more spans (runs of identical formatting).
        for line in block["lines"]:
            for span in line["spans"]:

                # Keep spans that contain visible text.
                if span["text"].strip():
                    spans.append((round(span["size"], 1), span["text"].strip()))

    # No text on this page.
    if not spans:
        return None, []

    # The most common font size is almost always the body text size.
    from collections import Counter
    body_size = Counter(size for size, _ in spans).most_common(1)[0][0]

    # Anything meaningfully larger than body text is probably a heading.
    headings = [t for size, t in spans if size > body_size * size_threshold_ratio]

    # Return the body size (for context) and the detected headings.
    return body_size, headings


# Only run if PyMuPDF is available.
if pymupdf is not None:

    # Open the GDPR - real legislation, where finding Article headings matters.
    document = pymupdf.open(gdpr_path)

    # Page 32 sits inside the articles, past the long recitals.
    body_size, headings = detect_headings_by_size(document.load_page(31))

    # Report what size-based detection found.
    print(f"GDPR page 32, body text size: {body_size}pt")
    print(f"Headings found by SIZE: {len(headings)}")

    # Close the document.
    document.close()

**Zero headings.** The size heuristic — which works fine on the arXiv papers — finds nothing
at all on real legislation.

Let's find out why before assuming the code is broken.

In [ ]:
# Only run if PyMuPDF is available.
if pymupdf is not None:

    # Reopen the GDPR.
    document = pymupdf.open(gdpr_path)

    # Inspect the actual span formatting on that page.
    structure = document.load_page(31).get_text("dict")

    # Count how many spans use each (size, font) combination.
    from collections import Counter
    combos = Counter()

    # Walk the structure collecting formatting signatures.
    for block in structure["blocks"]:
        if "lines" not in block:
            continue
        for line in block["lines"]:
            for span in line["spans"]:
                if span["text"].strip():
                    combos[(round(span["size"], 1), span["font"])] += 1

    # Show what formatting the page actually uses.
    print("size / font combinations on GDPR page 32:")
    print()
    for (size, font), count in combos.most_common():
        print(f"  {count:3d} spans   {size:5.1f}pt   {font}")

    # Close the document.
    document.close()

> **There is the answer.** Every span is **9.6pt** — headings are the *same size* as body
> text. They are distinguished only by the font name ending in `-Bold`.
>
> A size-based detector cannot possibly work here. This is not a bug in our code; it is a
> wrong assumption about how documents mark structure.

Legal and financial documents very often behave this way. Let's detect weight instead.

In [ ]:
# Define a function that detects headings by font WEIGHT as well as size.
def detect_headings(page, size_threshold_ratio=1.15):

    # Get the full span-level structure of the page.
    structure = page.get_text("dict")

    # Collect every span with the formatting attributes we care about.
    spans = []

    # Walk blocks -> lines -> spans.
    for block in structure["blocks"]:

        # Skip image blocks.
        if "lines" not in block:
            continue

        for line in block["lines"]:
            for span in line["spans"]:

                # Skip whitespace-only spans.
                if not span["text"].strip():
                    continue

                # PyMuPDF packs style bits into "flags"; bit 4 (value 16) means bold.
                is_bold = bool(span["flags"] & 2 ** 4)

                # Store size, boldness and the text itself.
                spans.append({
                    "size": round(span["size"], 1),
                    "bold": is_bold,
                    "text": span["text"].strip(),
                })

    # Nothing to analyse.
    if not spans:
        return []

    # The most common size is the body text size.
    from collections import Counter
    body_size = Counter(s["size"] for s in spans).most_common(1)[0][0]

    # A heading is EITHER noticeably larger OR bold at body size.
    headings = []

    # Classify each span.
    for span in spans:

        # Larger than body text - a heading in most documents.
        larger = span["size"] > body_size * size_threshold_ratio

        # Bold at body size - how legal and financial documents mark headings.
        bold_at_body_size = span["bold"] and span["size"] >= body_size

        # Keep it if either signal fires.
        if larger or bold_at_body_size:
            headings.append({
                "text": span["text"],
                "why": "size" if larger else "bold",
            })

    # Return the detected headings with the reason each was selected.
    return headings


# Only run if PyMuPDF is available.
if pymupdf is not None:

    # Reopen the GDPR.
    document = pymupdf.open(gdpr_path)

    # Run the improved detector on the same page that previously found nothing.
    headings = detect_headings(document.load_page(31))

    # Report the result.
    print(f"GDPR page 32 - headings found by SIZE+WEIGHT: {len(headings)}")
    print()

    # Show them.
    for heading in headings:
        print(f"  [{heading['why']}]  {heading['text'][:60]}")

    # Close the document.
    document.close()

Those are **real GDPR Article headings** — `Subject-matter and objectives` is Article 1,
`Material scope` is Article 2, `Territorial scope` is Article 3.

We recovered the structure of a piece of EU legislation using **font weight alone** — no ML
model, no layout service, no API call. That structure is exactly what
[module 2d](../2d.%20DOCX%20HTML%20and%20markdown/) gets for free from DOCX, and what Chapter 3 uses for
structure-aware chunking.

### The transferable lesson

Our first detector was not *buggy* — it encoded an assumption ("headings are bigger") that
happened to hold for arXiv papers and fail for legislation. We only found out by running it
on a document from a different domain.

> **Always test a heuristic on a document type you did not design it for.**

# 4. What text extraction silently misses

The IRS W-4 is an **AcroForm** — an interactive PDF with fillable fields. Form field values
are stored in a completely separate structure from the page text.

If you extract only text, you get the *labels* but never the *answers*. For a RAG system over
filled-in forms, that is a total failure — and a silent one.

In [ ]:
# Only run if PyMuPDF is available.
if pymupdf is not None:

    # Open the real IRS form.
    document = pymupdf.open(w4_path)

    # Count interactive form fields across the whole document.
    total_fields = 0

    # Collect a few field names to display.
    sample_fields = []

    # Walk every page looking for widgets (form fields).
    for page_number in range(len(document)):

        # Load the page.
        page = document.load_page(page_number)

        # page.widgets() yields interactive form fields.
        for widget in page.widgets():

            # Count it.
            total_fields += 1

            # Keep the first few names for display.
            if len(sample_fields) < 6:
                sample_fields.append((page_number + 1, widget.field_name, widget.field_type_string))

    # How much plain text the same document yields.
    text_length = sum(len(document.load_page(i).get_text("text"))
                      for i in range(len(document)))

    # Close the document.
    document.close()

    # Report both channels.
    print(f"Plain text extracted : {text_length:,} characters")
    print(f"Form fields found    : {total_fields}")
    print("\nSample fields (invisible to get_text):")
    for page_number, name, field_type in sample_fields:
        print(f"  p{page_number}  [{field_type:10s}] {name}")

> **This is the lesson of module 2b in one output.** A parser that reports
> "extraction successful, 4,800 characters" has silently discarded every form field. Nothing
> errored. Nothing warned. The data is simply absent from your index.

The same applies to: annotations and comments, embedded attachments, layers, JavaScript
actions, and alt-text on images.

**The general rule:** ask what channels a format has, and confirm you read the ones that
carry meaning — do not assume `get_text()` is exhaustive.

# 5. Module summary

## Key points

1. **PDF stores coordinates, not meaning.** All structure is inferred.
2. **Classify before extracting**: digital → parser, scanned → OCR, mixed → both.
3. **Preserve `page_number`** through extraction, or citations become impossible.
4. **Reading order is inferred** — verify it, especially on multi-column layouts.
5. **Font size recovers heading structure** cheaply, with no ML.
6. **`get_text()` is not exhaustive** — form fields, annotations and attachments are separate channels.

## Interview answer: "Why is PDF parsing difficult?"

> PDF is a presentation-oriented format. It often stores text as positioned glyphs rather
> than explicit semantic structures such as paragraphs and tables. Multi-column layouts,
> scanned pages, headers, footers, tables, figures, and reading-order ambiguity make
> extraction difficult. Interactive form fields and annotations live in entirely separate
> structures that plain text extraction never sees. Production RAG therefore requires
> layout-aware parsers, OCR fallback, structural reconstruction, and extraction-quality
> validation.

## Exercises

1. **Break the column detector.** Find a page in the RAG paper with a full-width figure or
   table spanning both columns. Does the midpoint split corrupt it? How would you fix it?
2. **Benchmark two parsers.** Install `pdfplumber` and extract GDPR page 40 with both.
   Compare character counts, then compare *quality*. Which "extracted more"? Which is better?
3. **Extract the W-4 tables.** Use `page.find_tables()` and compare against `get_text()`.
   What relationships does plain text lose?
4. **Build a heading extractor** for the GDPR that finds every `Article N` heading with its
   page number — the index a legal RAG system needs.


## → Next: [Module 2c — OCR & Scanned Documents](../2c.%20OCR%20and%20scanned%20documents/)

When there is no text layer at all, you need OCR. We run real Tesseract over real scan
images, read genuine per-word confidence scores, and build the routing logic that decides
which documents deserve the expensive path.